In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
import random
import time

import networkx as nx
from numpy import linalg as LA
from numpy.random import rand

from scipy.integrate import solve_ivp

In [2]:
#Auxiliary functions

#convert a Laplacian with free boundary conditions to one with fixed boundary conditions
def to_fixed(L):
    L_max=np.max(L)
    for i in range (len(L)):
        L[i][i]=L_max
    return L

#build Laplacian with fixed boundary conditions for a string with N nodes
def build_L_1(N):
    G = nx.grid_graph(dim=(N,1),periodic=False)
    G=nx.convert_node_labels_to_integers(G)
    L=nx.laplacian_matrix(G).toarray()
    L=to_fixed(L)
    return L

#build auxiliary matrix
def build_L_2(N):
    L=np.zeros((N,N))
    for i in range (N):
        L[i][(i+1)%N]=1
        L[i][(i-1)%N]=-1
    L[0][N-1]=0
    L[N-1][0]=0
    return L

#Build first damping dependent matrix for the Jacobian
def M_1(beta):
    M=np.zeros((N,N))
    for i in range (N):
        M[i][i]=beta[i]*L_1[i][i]
        M[i][(i+1)%N]=beta[i]*L_1[i][(i+1)%N]
        M[i][(i-1)%N]=beta[i]*L_1[i][(i-1)%N]
        
    return M

#Build second damping dependent matrix for the Jacobian
def M_2(beta):
    M=np.zeros((N,N))
    for i in range (1,N-1):
        M[i][(i+1)%N]=(beta[(i+1)%N]-beta[(i-1)%N])*L_2[i][(i+1)%N]
        M[i][(i-1)%N]=(beta[(i+1)%N]-beta[(i-1)%N])*L_2[i][(i-1)%N]
    
    M[0][1]=-3*beta[0]+4*beta[1]-beta[2]
    M[N-1][N-2]=-3*beta[N-1]+4*beta[N-2]-beta[N-3]
    return M

In [ ]:
#Jacobian
def Jac(x):
    M_beta_1=M_1(x)
    M_beta_2=M_2(x)
    return np.block([[np.zeros((N,N)),np.identity(N)],[-L_1/dx**2,-2*M_beta_1/dx**2-M_beta_2/(2*dx**2)]])

#Maximum Lyapunov exponent
def objective(x):
    J=Jac(x)
    EV=np.real(np.sort(LA.eigvals(J)))
    return EV[len(EV)-1]

In [3]:


#Compare the decay from a damping derived from the eigenmode analysis and the optimal homogeneous
N=100
dx=np.pi/N
L_1=build_L_1(N)
L_2=build_L_2(N)

x=np.linspace(0,np.pi,N)
b_optimized= 0.10692908*(3.40984784-0.11545739*np.cos(x)-1.40131319*np.cos(2*x)-1.20896208*np.cos(3*x)+ 0.34234185*np.cos(4*x))
b_hom=[np.sqrt(2)/2]*N


#First, compare their decay rates
print(objective(b_optimized))
print()
print(objective(b_hom))

-0.8077462780029812

-0.6931180979808059


In [14]:
start=time.time()

#Triangular initial condition
u_ini=np.zeros(2*N)
x_vec=np.linspace(0,np.pi,N)
for i in range (N//3):
    u_ini[i+N]=3/(2*np.pi)*x_vec[i]
    u_ini[i+N+N//3]=1/2+3/(2*np.pi)*x_vec[i]
    u_ini[i+N+2*N//3]=1-3*x_vec[i]/np.pi

dx=np.pi/N
dt_max=10**(-2)#dt
    
L_1=build_L_1(N)
L_2=build_L_2(N)

t_f=10

def dynamics(t,y):
    return np.matmul(J,y)


#Evolve the system with the heterogeneous damping
b=b_optimized
M_beta_1=M_1(b)
M_beta_2=M_2(b)
J=np.block([[np.zeros((N,N)),np.identity(N)],[-L_1/dx**2,-2*M_beta_1/dx**2-M_beta_2/(2*dx**2)]])
sol_opt = solve_ivp(dynamics, [0,t_f], u_ini, method='DOP853',max_step=dt_max, dense_output=False)


#Evolve the system with the optimal homogeneous damping
b=b_hom
M_beta_1=M_1(b)
M_beta_2=M_1(b)
J=np.block([[np.zeros((N,N)),np.identity(N)],[-L_1/dx**2,-2*M_beta_1/dx**2-M_beta_2/(2*dx**2)]])
sol_hom = solve_ivp(dynamics, [0,t_f], u_ini, method='DOP853',max_step=dt_max, dense_output=False)


end=time.time()
print()
print(end-start)

KeyboardInterrupt: 

In [ ]:
#Extract the distance to equilibrium as a function of time from the solution

sol_opt=np.transpose(sol_opt.y)
sol_hom=np.transpose(sol_hom.y)

t_tot_opt=len(sol_opt)
t_tot_hom=len(sol_hom)

dist_opt=np.zeros(t_tot_opt)
dist_hom=np.zeros(t_tot_hom)
for i in range (t_tot_opt):
    dist_opt[i]=LA.norm(sol_opt[i])
    
for i in range (t_tot_hom):
    dist_hom[i]=LA.norm(sol_hom[i])

In [ ]:
#Plot the distance to equilibrium as a function of time

tspan_opt=np.linspace(0,t_f,t_tot_opt)
tspan_hom=np.linspace(0,t_f,t_tot_hom)

fig,ax=plt.subplots(figsize=(20,10))

plt.plot(tspan_opt, dist_opt, color='red', linewidth=2,label='Optimal Heterogeneous assymetric damping')
plt.plot(tspan_hom, dist_hom, color='blue', linewidth=2,label='Optimal Homogeneous symmetric damping')


ax.tick_params(axis='both', which='major', labelsize=30)
ax.tick_params(axis='both', which='minor', labelsize=28)
plt.xlabel('Time',size=40)
plt.ylabel('Energy',size=40)
ax.set_yscale('log')
plt.legend(fontsize=22)
plt.show()